# 09 — InceptionV3 pipeline on a Colab GPU runtime (attached from VSCode)

Use this when you've connected VSCode to a **Colab GPU runtime**. Important: the kernel and filesystem live on Colab's remote VM, **not** your Mac — so your local repo files aren't visible here. This notebook therefore clones the repo and rebuilds the dataset *on the VM*, then runs the whole InceptionV3 pipeline on `--device cuda`.

Difference from `08_colab_inception_pipeline.ipynb` (which is meant for the Colab web UI): this one **retrains the model on the GPU** by default, avoiding the Google-Drive auth popup that's unreliable over a VSCode-attached runtime. Drive is offered as an optional alternative if you want to reuse your exact checkpoint.

Run cells top-to-bottom. See [`docs/inception_v3_experiment.md`](https://github.com/1minute99/DSC291_Research_Reproduction/blob/inception-extension/docs/inception_v3_experiment.md) for what each stage means.

In [ ]:
# 0. Confirm the remote kernel actually has a GPU.
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- attach a GPU runtime!')

In [ ]:
# 1. Clone the repo + MILAN submodule onto the Colab VM.
REPO_URL = 'https://github.com/1minute99/DSC291_Research_Reproduction.git'
BRANCH   = 'inception-extension'
REPO     = '/content/DSC291_Research_Reproduction'

import os
if not os.path.isdir(REPO):
    !git clone --recurse-submodules --branch {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git submodule update --init --recursive
!git log --oneline -1

In [ ]:
# 2. Install deps. KEEP Colab's preinstalled CUDA torch/torchvision (don't let
#    milan/requirements.in's CPU pins override them); SKIP allennlp (the repo's
#    _shims/allennlp handles it via milan_glue/upstream.py).
!grep -ivE '^[[:space:]]*(allennlp|torch|torchvision)([[:space:]<>=!]|$)' milan/requirements.in > /tmp/req_milan_colab.txt
!pip install -q -r /tmp/req_milan_colab.txt
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 3. Environment variables + import paths (mirror environment.md).
import os, sys
os.environ['MILAN_DATA_DIR']    = f'{REPO}/data'
os.environ['MILAN_MODELS_DIR']  = f'{REPO}/models'
os.environ['MILAN_RESULTS_DIR'] = f'{REPO}/results'
os.environ['PYTHONPATH']        = f'{REPO}:{REPO}/milan'   # for `!python -m ...`
for p in (REPO, f'{REPO}/milan'):
    if p not in sys.path:
        sys.path.insert(0, p)
print('env set')

In [ ]:
# 4. Rebuild the spurious dataset (deterministic, seed=0 -> identical to local).
#    Downloads Imagenette (~325 MB) on first run.
!python -m milan_repro.data.build_splits

In [ ]:
# 5. Train InceptionV3 on the GPU (default path -- self-contained, no Drive).
#    ~minutes on a T4/A100. Writes models/inception_v3_spurious.pth.
!python -m milan_repro.train.train_inception_v3 --device cuda

> **Optional — reuse your exact local checkpoint instead of retraining.** If Drive auth works over your VSCode connection, upload `inception_v3_spurious.pth` to `MyDrive/dsc291/` and run:
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> !cp /content/drive/MyDrive/dsc291/inception_v3_spurious.pth models/
> ```
> Then skip cell 5.

In [ ]:
# 6. Exemplars (dissection): top-k activating regions for all ~3,936 units at 299px.
!python -m milan_repro.milan_glue.run_exemplars --arch inception_v3 --device cuda

In [ ]:
# 7. Descriptions: MILAN `base` decoder captions each unit (downloads ~1 GB
#    decoder + ResNet101 encoder weights on first run).
!python -m milan_repro.milan_glue.run_descriptions \
    --dissect-dir results/edit/imagenet-spurious-text/inception_v3_spurious-50pct \
    --out results/descriptions_inception_v3.csv --device cuda

In [ ]:
# 8. Flag text neurons (description contains text/word/letter).
!python -m milan_repro.editing.identify_text_neurons \
    --descriptions results/descriptions_inception_v3.csv \
    --out results/descriptions_inception_v3_annotated.csv

In [ ]:
# 9. Editing experiment: per-unit importance + text-sorted / sort-all / random
#    curves. The slow stage on MPS; far faster on CUDA. Raise --ablation-step /
#    lower --ablation-max to trim further.
!python -m milan_repro.editing.evaluate --arch inception_v3 --device cuda

In [ ]:
# 10. Figures + recovery summary.
!python -m milan_repro.figures.plot_fig7 \
    --dissect-dir results/edit/imagenet-spurious-text/inception_v3_spurious-50pct \
    --descriptions results/descriptions_inception_v3_annotated.csv \
    --out results/figs/fig7_inception_v3.pdf
!python -m milan_repro.figures.plot_fig8 \
    --ablation-csv results/ablation_curve_inception_v3.csv \
    --out results/figs/fig8_inception_v3.pdf \
    --title 'InceptionV3 robustness vs. ablation (MILAN Section 7, extension)'

import pandas as pd
df = pd.read_csv('results/ablation_curve_inception_v3.csv')
base = df[df['mode']=='baseline'].iloc[0]['adv_acc']
best = df[df['mode'].isin(['text-sorted','sort-all'])].groupby('mode')['adv_acc'].max()
print('adv baseline:', round(base,4))
print('text-sorted recovery: +{:.1f} pt'.format(100*(best.get('text-sorted',float('nan'))-base)))
print('sort-all   recovery: +{:.1f} pt'.format(100*(best.get('sort-all',float('nan'))-base)))

In [ ]:
# 11. Bundle the small result artifacts so you can pull them back to your Mac.
#     In VSCode: right-click the .zip in the remote Explorer -> Download,
#     or copy it to Drive if mounted.
!cd results && zip -q -r /content/inception_results.zip \
    descriptions_inception_v3.csv descriptions_inception_v3_annotated.csv \
    ablation_curve_inception_v3.csv importance_inception_v3.csv figs 2>/dev/null
!ls -lh /content/inception_results.zip